# 12.6 System-level failure attribution

The components are each sound in isolation; the system question is what drives failure when they run together. The framework answers it two ways. The actionable analysis is **per component** -- the attribution of Chapter~10 run on each tool's own correctness, so it is clear **which component to fix**. Because the failures are few (nine of ninety-two classifier runs, one of eighteen flag runs) a logistic fit is unstable and its odds ratios can diverge at zero-failure levels, so the robust report is the **empirical failure rate** at each factor level. The per-query **decision** is kept as a separate **overall** summary plus the weak-link decomposition.

**Reference (run separately):** the calls that produced these numbers. The seed case is excluded from each attribution - it was a blocking factor, not a property under test.

```python
# Reference only - NOT executed here (needs agentlab/gmstest, models, a store).
from gmstest import evaluate

# PER-COMPONENT: which input breaks which tool (empirical per-level failure rates).
# search_policy is graded by graph-truth (Sec. 12.5), so it is not label-attributed here.
for col in ('c_classify_complaint', 'c_flag_regulatory'):
    sub = [r for r in rows if r[col] is not None]
    evaluate.attribute(sub, suite.factor_names, metric=col)

# OVERALL per-query decision (right classification AND escalation) + weak-link.
overall = evaluate.attribute(rows, suite.factor_names)   # metric defaults to 'correct'
blame   = evaluate.weak_link(rows, workflow)
```

**What the next cell does** -- load the pinned artifact and define a table printer:

1. **Find the artifact.** Walk up from `Path.cwd()` to `data/capstone_run.json` and `json.loads` it into `D`.
2. **Pull the views.** `by_component = D['attribution_by_component']` (per-tool empirical rates), `overall = D['overall']` (the conjunction) and `weak_link = D['weak_link']`.
3. **Define `show_factor_table`.** Prints the overall logistic attribution -- one row per factor (`G2`, `p_adj`, `pseudo_r2`, `significant_adj`) with the worst level by odds ratio for a significant factor.

In [ ]:
import json
from pathlib import Path

root = Path.cwd().parent / 'code'
while not (root/'data'/'capstone_run.json').exists() and root != root.parent:
    root = root.parent
D = json.loads((root/'data'/'capstone_run.json').read_text())
by_component = D['attribution_by_component']
overall = D['overall']
weak_link = D['weak_link']
WORKFLOW = ['classify_complaint', 'extract_facts', 'search_policy',
            'flag_regulatory', 'draft_response']
FACTOR_ORDER = ['reasoning_cue', 'clarity', 'entity_aliasing']

def show_factor_table(tbl, indent='  '):
    print(f"{indent}{'factor':16s}{'G2':>8s}{'p_adj':>9s}{'pseudoR2':>10s}{'sig':>6s}   worst level (odds ratio)")
    for r in tbl:
        ors = r['odds_ratios']; worst = min(ors, key=ors.get)
        worst_str = f"{worst} ({ors[worst]})" if r['significant_adj'] else '---'
        print(f"{indent}{r['factor']:16s}{r['G2']:>8.2f}{r['p_adj']:>9.4f}{r['pseudo_r2']:>10.3f}"
              f"{str(r['significant_adj']):>6s}   {worst_str}")
print('loaded per-component + overall attribution from', root/'data')

## Which input breaks which component

Each label-scored tool is analyzed on its own correctness, on the runs where it was graded. For each factor we report the **failure rate at each level** -- failed / graded -- which never diverges, unlike a logistic odds ratio on so few failures. If a level concentrates the failures it is the input condition to fix for that tool; if the rates are flat, no presentation factor drives the tool. (`search_policy` is graded against the graph ground truth in Section 12.5, not by a label match, so it does not enter this label-attribution loop.)

**What the next cell does** -- print each label-scored tool's per-level failure rates:

1. **Loop the two label-scored tools.** For each, read `by_component[tool]` and print its `scored` and `failures`.
2. **Per factor, per level.** For each factor (reasoning cue first) print `level=failed/n` across its levels.
3. **No fit forced.** The rates stand on their own; a clean tool simply shows zero failures.

In [ ]:
for tool in ('classify_complaint', 'flag_regulatory'):
    a = by_component[tool]
    print(f"{tool}  (graded {a['scored']}, failures {a['failures']})")
    for f in FACTOR_ORDER:
        levels = a['by_factor'][f]
        cells_str = '  '.join(f"{lvl}={v['failed']}/{v['n']}" for lvl, v in levels.items())
        print(f"    {f:16s}{cells_str}")
    print()

The reading is direct. The classifier's nine failures spread across the levels: the nearest thing to a gradient is `clarity` (ambiguous 5/22 against clear 1/35), and the reasoning cue is flat (four failures with a `misleading_cue`, four without). No level concentrates the failures, and with only nine the logistic deviance test certifies nothing after correction (`clarity` is the nearest at $p_{adj}=0.96$). This reverses the finding on the previous agent, where a downplaying cue was the dominant failure driver; the retrained classifier no longer breaks on the presentation factors the design varies. The flagger fails once in eighteen and has nothing to attribute.

## The overall decision, as a summary

Read as a whole, the question is whether the agent reached the right decision on each query -- the right classification and the right escalate/don't-escalate call. Its rate, its logistic factor attribution and the weak-link blame say how often the pipeline decides right and where it breaks. The per-tool numbers are scored separately, not folded in here.

**What the next cell does** -- summarize the decision:

1. **Rate.** Print `overall['accuracy']` and its `definition`.
2. **Attribution.** `show_factor_table(overall['attribution'])` -- the logistic fit on 120 runs.
3. **Weak-link.** Walk `WORKFLOW` and print each tool's blame count from `weak_link`, then the total.

In [ ]:
print(f"Overall decision accuracy: {overall['accuracy']:.3f}")
print(f"  {overall['definition']}\n")
print('Overall factor attribution (logistic, BH-corrected)')
show_factor_table(overall['attribution'])
print()
print('Weak-link blame (wrongly-decisioned runs -> first erring tool)')
for tool in WORKFLOW:
    if tool in weak_link:
        print(f"  {tool:20s} {weak_link[tool]}")
print(f"  {'total tool-attributed':20s} {sum(weak_link.values())}")

The agent decides right on about two runs in three (0.675). Run on the decision bit, the factor attribution finds **nothing** significant after correction, which agrees with the per-component view: the retrained agent does not break on the presentation factors the design varies. The weak-link decomposition charges the wrongly-decisioned runs to the first tool that erred -- nine to `classify_complaint`, one to the flagger; the rest carry no erring graded step and turn on the end decision itself, a property of the whole trajectory. There is no single input condition to harden against here: the failures are few and spread.

## Where the 81 correctly-decisioned and 39 wrong runs come from

The decision collapses each run to a single bit, so it is worth seeing *which* runs make up the 81 that decide right and the 39 that do not. A run is judged on the steps that applied to it, plus the end escalate/don't-escalate decision, and it is never required to reach every tool. A short inquiry is scored on classification alone; a complaint that implicates no regulation is scored on classification and policy search; only a regulation-implicating complaint is also scored on `flag_regulatory`; and the PII and prompt-injection inputs are intercepted by the policy gate before any component is graded. That is why the flagger is graded on just 18 of the 120 runs.

**What the next cell does** -- load the per-run rows (`capstone_rows.json`), group every run by the steps that were actually graded on it, and then trace the failing side: how many failures a graded component owns versus how many turn on the end decision alone.

In [ ]:
# Where the 81 correctly-decisioned and the 39 wrong runs come from -- per run, from the pinned rows.
rows = json.loads((root/'data'/'capstone_rows.json').read_text())['rows']
ccols = ['c_classify_complaint', 'c_search_policy', 'c_flag_regulatory']

def graded(r):                      # the scored steps that actually applied to this run
    return [c[2:] for c in ccols if r[c] is not None]

def path(r):                        # name the route by the last step that was graded
    g = graded(r)
    if not g:                       return 'escalated / refused (no component graded)'
    if 'flag_regulatory' in g:      return 'complaint implicating a regulation'
    if 'search_policy'  in g:       return 'complaint, no regulation'
    return 'inquiry / other (classify only)'

from collections import Counter
ORDER = ['inquiry / other (classify only)', 'complaint, no regulation',
         'complaint implicating a regulation', 'escalated / refused (no component graded)']
runs_by = Counter(path(r) for r in rows)
ok_by   = Counter(path(r) for r in rows if r['correct'] == 1)

print(f"{'path the run took':48s}{'runs':>6s}{'correct':>9s}")
for p in ORDER:
    print(f"  {p:46s}{runs_by[p]:>6d}{ok_by[p]:>9d}")
print(f"  {'TOTAL':46s}{sum(runs_by.values()):>6d}{sum(ok_by.values()):>9d}")

# The failing side, traced.
fail     = [r for r in rows if r['correct'] == 0]
comp_err = [r for r in fail if any(r[c] == 0 for c in ccols)]   # a graded step was wrong
end_only = [r for r in fail if r not in comp_err]               # graded steps ok; end decision wrong
esc_path = [r for r in fail if not graded(r)]                   # took the escalation / refusal path
print(f"\nfailing runs: {len(fail)}")
print(f"  a graded component erred           : {len(comp_err):2d}   weak-link {D['weak_link']}")
print(f"  graded steps OK, end decision wrong: {len(end_only):2d}")
print(f"    of those, on the escalation path : {len(esc_path):2d}   (nothing graded -> rides on the end decision)")

# Structure only -- bounds the partition, not the volatile point estimates.
assert sum(runs_by[p] for p in ORDER) == len(rows) and set(runs_by) <= set(ORDER)
assert all(ok_by[p] <= runs_by[p] for p in ORDER)
assert sum(ok_by.values()) == sum(1 for r in rows if r['correct'] == 1)
assert len(comp_err) + len(end_only) == len(fail)

Read down the table. Only **18** runs are regulation-implicating complaints, so `flag_regulatory` is graded on 18; the other correctly-decisioned runs decide on classification alone (23 of 32) or on classification plus policy search (41 of 42), and never need the flagger. The 81 correct are therefore 23 + 41 + 17.

The **28** PII and prompt-injection inputs are intercepted at the policy gate before a classification is emitted, so they carry no matching taxonomy label and enter the joint criterion as zero; the gate handles them correctly, graded on its own in Section 12.5.

The failing side: of the 39 wrong runs, the weak-link decomposition charges **10** to the first tool that erred (nine to `classify_complaint`, one to the flagger). The remaining **29** have every graded step correct and turn on the end decision instead, and **28** of those took the gate-intercepted path, where no component is graded at all. That is the whole-trajectory outcome the decision bit folds in and the per-component view cannot see.

**What the next cell does** -- pin the SHAPE of the analysis (the campaign is reproducible on the pinned inputs, but the checks bound structure, not volatile point estimates): each tool's per-level failure rates sum to its failure and graded counts; the overall conjunction is a valid rate with a well-formed logistic table; weak-link blame falls only on workflow tools.

In [ ]:
FACTORS = {'clarity', 'entity_aliasing', 'reasoning_cue'}
assert set(by_component) == {'classify_complaint', 'search_policy', 'flag_regulatory'}
for tool, a in by_component.items():
    assert set(a['by_factor']) == FACTORS
    for f, levels in a['by_factor'].items():
        assert sum(v['failed'] for v in levels.values()) == a['failures']
        assert sum(v['n'] for v in levels.values()) == a['scored']
FACT_KEYS = {'factor', 'G2', 'p_adj', 'pseudo_r2', 'significant_adj', 'odds_ratios'}
assert 0.0 <= overall['accuracy'] <= 1.0
assert {r['factor'] for r in overall['attribution']} == FACTORS
for r in overall['attribution']:
    assert FACT_KEYS <= set(r) and 0.0 <= r['p_adj'] <= 1.0
assert set(weak_link) <= set(WORKFLOW) and all(v > 0 for v in weak_link.values())
print('self-check passed')